# Benchmark Evaluation for LLMs

> Implement standard benchmarks to evaluate LLM capabilities

**Benchmarks Covered:**
- **MMLU** — Massive Multitask Language Understanding (57 subjects)
- **GSM8K** — Grade School Math (8.5K problems)
- **HumanEval** — Code generation (164 problems)
- **TruthfulQA** — Truthfulness (817 questions)
- **HellaSwag** — Commonsense reasoning (40K problems)
- **ARC** — Science questions (7.8K problems)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from collections import defaultdict
import json
import re

## 1. Benchmark Data Structures

In [ ]:
@dataclass
class BenchmarkQuestion:
    """Standard benchmark question format"""
    id: str
    question: str
    choices: List[str]
    correct_answer: int  # Index of correct choice
    subject: str = "general"
    difficulty: str = "medium"

@dataclass
class CodeProblem:
    """Coding benchmark problem"""
    id: str
    prompt: str
    test_cases: List[Tuple]
    canonical_solution: str

@dataclass
class MathProblem:
    """Math word problem"""
    id: str
    question: str
    answer: float
    reasoning: str = ""

# Sample MMLU-style questions
mmlu_samples = [
    BenchmarkQuestion(
        id="mmlu_001",
        question="What is the primary function of mitochondria in a cell?",
        choices=[
            "Protein synthesis",
            "Energy production",
            "Cell division",
            "Waste removal"
        ],
        correct_answer=1,
        subject="biology",
        difficulty="easy"
    ),
    BenchmarkQuestion(
        id="mmlu_002",
        question="Which of the following is NOT a fundamental force of nature?",
        choices=[
            "Gravity",
            "Electromagnetism",
            "Friction",
            "Strong nuclear force"
        ],
        correct_answer=2,
        subject="physics",
        difficulty="easy"
    ),
    BenchmarkQuestion(
        id="mmlu_003",
        question="In Python, what does the 'yield' keyword do?",
        choices=[
            "Returns a value and exits the function",
            "Pauses function execution and returns a generator",
            "Raises an exception",
            "Imports a module"
        ],
        correct_answer=1,
        subject="computer_science",
        difficulty="medium"
    ),
    BenchmarkQuestion(
        id="mmlu_004",
        question="What is the derivative of e^x?",
        choices=[
            "x * e^(x-1)",
            "e^x",
            "ln(x)",
            "1/x"
        ],
        correct_answer=1,
        subject="mathematics",
        difficulty="easy"
    ),
    BenchmarkQuestion(
        id="mmlu_005",
        question="Which economic concept describes the cost of the next best alternative foregone?",
        choices=[
            "Sunk cost",
            "Opportunity cost",
            "Marginal cost",
            "Fixed cost"
        ],
        correct_answer=1,
        subject="economics",
        difficulty="medium"
    ),
]

# Sample GSM8K-style math problems
gsm8k_samples = [
    MathProblem(
        id="gsm_001",
        question="Janet buys 3 pounds of broccoli for $4 per pound and 2 pounds of carrots for $2 per pound. How much does she spend?",
        answer=16.0,
        reasoning="3 × $4 = $12 for broccoli. 2 × $2 = $4 for carrots. Total = $12 + $4 = $16"
    ),
    MathProblem(
        id="gsm_002",
        question="A bakery made 240 cupcakes. They sold 3/4 of them. How many cupcakes remain?",
        answer=60.0,
        reasoning="3/4 of 240 = 180 sold. 240 - 180 = 60 remain"
    ),
    MathProblem(
        id="gsm_003",
        question="Tom has $50. He buys 3 books at $8 each and a pen for $5. How much money does he have left?",
        answer=21.0,
        reasoning="3 × $8 = $24 for books. $24 + $5 = $29 total. $50 - $29 = $21 left"
    ),
]

# Sample HumanEval-style coding problems
humaneval_samples = [
    CodeProblem(
        id="he_001",
        prompt="Write a function that returns the sum of two numbers.",
        test_cases=[
            ((2, 3), 5),
            ((-1, 1), 0),
            ((0, 0), 0),
        ],
        canonical_solution="def add(a, b):\n    return a + b"
    ),
    CodeProblem(
        id="he_002",
        prompt="Write a function that checks if a string is a palindrome.",
        test_cases=[
            (("racecar",), True),
            (("hello",), False),
            (("",), True),
        ],
        canonical_solution="def is_palindrome(s):\n    return s == s[::-1]"
    ),
]

print(f"MMLU samples: {len(mmlu_samples)}")
print(f"GSM8K samples: {len(gsm8k_samples)}")
print(f"HumanEval samples: {len(humaneval_samples)}")

## 2. Simulated Model Responses

In [ ]:
class SimulatedModel:
    """Simulate model responses for demonstration"""
    
    def __init__(self, accuracy: float = 0.7, name: str = "Model"):
        self.accuracy = accuracy
        self.name = name
        np.random.seed(hash(name) % 10000)
    
    def answer_mmlu(self, question: BenchmarkQuestion) -> int:
        """Simulate multiple-choice answer"""
        if np.random.random() < self.accuracy:
            return question.correct_answer
        else:
            wrong = [i for i in range(len(question.choices)) if i != question.correct_answer]
            return np.random.choice(wrong)
    
    def answer_gsm8k(self, problem: MathProblem) -> float:
        """Simulate math answer"""
        if np.random.random() < self.accuracy:
            return problem.answer
        else:
            # Return wrong answer with some noise
            return problem.answer * np.random.uniform(0.5, 1.5)
    
    def generate_code(self, problem: CodeProblem) -> str:
        """Simulate code generation"""
        if np.random.random() < self.accuracy:
            return problem.canonical_solution
        else:
            return "def solution():\n    pass"  # Incorrect stub

# Create models with different capabilities
models = [
    SimulatedModel(accuracy=0.9, name="GPT-4")
    SimulatedModel(accuracy=0.75, name="GPT-3.5")
    SimulatedModel(accuracy=0.6, name="Llama-2-7B")
    SimulatedModel(accuracy=0.45, name="Small-Model")
]

print("Models initialized:")
for m in models:
    print(f"  {m.name}: {m.accuracy:.0%} accuracy")

## 3. MMLU Evaluation

In [ ]:
def evaluate_mmlu(model: SimulatedModel, questions: List[BenchmarkQuestion]) -> Dict:
    """Evaluate model on MMLU-style questions"""
    correct = 0
    subject_correct = defaultdict(lambda: [0, 0])  # [correct, total]
    difficulty_correct = defaultdict(lambda: [0, 0])
    
    results = []
    
    for q in questions:
        pred = model.answer_mmlu(q)
        is_correct = (pred == q.correct_answer)
        
        correct += int(is_correct)
        subject_correct[q.subject][0] += int(is_correct)
        subject_correct[q.subject][1] += 1
        difficulty_correct[q.difficulty][0] += int(is_correct)
        difficulty_correct[q.difficulty][1] += 1
        
        results.append({
            'id': q.id,
            'correct': is_correct,
            'predicted': pred,
            'actual': q.correct_answer
        })
    
    # Calculate metrics
    accuracy = correct / len(questions)
    
    subject_acc = {
        sub: corr / total
        for sub, (corr, total) in subject_correct.items()
    }
    
    difficulty_acc = {
        diff: corr / total
        for diff, (corr, total) in difficulty_correct.items()
    }
    
    return {
        'accuracy': accuracy,
        'correct': correct,
        'total': len(questions),
        'subject_accuracy': subject_acc,
        'difficulty_accuracy': difficulty_acc,
        'results': results
    }

# Evaluate all models on MMLU
mmlu_results = {}
for model in models:
    mmlu_results[model.name] = evaluate_mmlu(model, mmlu_samples)
    print(f"\n{model.name} MMLU: {mmlu_results[model.name]['accuracy']:.1%}")
    for sub, acc in mmlu_results[model.name]['subject_accuracy'].items():
        print(f"  {sub}: {acc:.1%}")

## 4. GSM8K Evaluation

In [ ]:
def evaluate_gsm8k(model: SimulatedModel, problems: List[MathProblem],
                   tolerance: float = 0.01) -> Dict:
    """Evaluate model on math word problems"""
    correct = 0
    exact_match = 0
    errors = []
    
    for p in problems:
        pred = model.answer_gsm8k(p)
        is_correct = abs(pred - p.answer) < tolerance
        
        correct += int(is_correct)
        exact_match += int(abs(pred - p.answer) < 1e-6)
        
        if not is_correct:
            errors.append({
                'id': p.id,
                'predicted': pred,
                'actual': p.answer,
                'error': abs(pred - p.answer)
            })
    
    return {
        'accuracy': correct / len(problems),
        'exact_match': exact_match / len(problems),
        'correct': correct,
        'total': len(problems),
        'mean_error': np.mean([e['error'] for e in errors]) if errors else 0,
        'errors': errors
    }

# Evaluate GSM8K
gsm8k_results = {}
for model in models:
    gsm8k_results[model.name] = evaluate_gsm8k(model, gsm8k_samples)
    r = gsm8k_results[model.name]
    print(f"{model.name} GSM8K: {r['accuracy']:.1%} (exact: {r['exact_match']:.1%})")

## 5. HumanEval Evaluation

In [ ]:
def evaluate_humaneval(model: SimulatedModel, problems: List[CodeProblem]) -> Dict:
    """Evaluate model on coding problems"""
    passed = 0
    total_tests = 0
    problem_passed = 0
    
    results = []
    
    for p in problems:
        code = model.generate_code(p)
        
        # Simulate test execution
        # In practice: use sandboxed execution
        tests_passed = 0
        for inputs, expected in p.test_cases:
            total_tests += 1
            # Simulate: if code is correct solution, all tests pass
            if code == p.canonical_solution:
                tests_passed += 1
            else:
                # Random chance for partial credit
                tests_passed += int(np.random.random() < 0.1)
        
        all_passed = (tests_passed == len(p.test_cases))
        problem_passed += int(all_passed)
        passed += tests_passed
        
        results.append({
            'id': p.id,
            'tests_passed': tests_passed,
            'total_tests': len(p.test_cases),
            'all_passed': all_passed
        })
    
    return {
        'pass_at_1': problem_passed / len(problems),
        'test_accuracy': passed / total_tests,
        'problems_solved': problem_passed,
        'total_problems': len(problems),
        'results': results
    }

# Evaluate HumanEval
humaneval_results = {}
for model in models:
    humaneval_results[model.name] = evaluate_humaneval(model, humaneval_samples)
    r = humaneval_results[model.name]
    print(f"{model.name} HumanEval: {r['pass_at_1']:.1%} pass@1, {r['test_accuracy']:.1%} test acc")

## 6. Comprehensive Results Dashboard

In [ ]:
# Aggregate all results
all_results = {}
for model in models:
    all_results[model.name] = {
        'MMLU': mmlu_results[model.name]['accuracy'] * 100,
        'GSM8K': gsm8k_results[model.name]['accuracy'] * 100,
        'HumanEval': humaneval_results[model.name]['pass_at_1'] * 100,
    }

# Create comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

model_names = [m.name for m in models]
benchmarks = ['MMLU', 'GSM8K', 'HumanEval']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

# 1. Overall comparison bar chart
x = np.arange(len(model_names))
width = 0.25

for i, bench in enumerate(benchmarks):
    values = [all_results[m][bench] for m in model_names]
    axes[0, 0].bar(x + i * width, values, width, label=bench, color=colors[i])

axes[0, 0].set_ylabel('Accuracy (%)')
axes[0, 0].set_title('Benchmark Comparison Across Models')
axes[0, 0].set_xticks(x + width)
axes[0, 0].set_xticklabels(model_names, rotation=15, ha='right')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Radar chart
angles = np.linspace(0, 2 * np.pi, len(benchmarks), endpoint=False).tolist()
angles += angles[:1]

ax_radar = fig.add_subplot(2, 2, 2, projection='polar')
for i, model_name in enumerate(model_names):
    values = [all_results[model_name][b] for b in benchmarks]
    values += values[:1]
    ax_radar.plot(angles, values, 'o-', linewidth=2, label=model_name)
    ax_radar.fill(angles, values, alpha=0.1)

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(benchmarks)
ax_radar.set_ylim(0, 100)
ax_radar.set_title('Capability Radar', pad=20)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

# 3. MMLU subject breakdown
subjects = list(mmlu_results[model_names[0]]['subject_accuracy'].keys())
for i, model_name in enumerate(model_names):
    accs = [mmlu_results[model_name]['subject_accuracy'].get(s, 0) * 100
            for s in subjects]
    axes[1, 0].plot(subjects, accs, marker='o', label=model_name, linewidth=2)

axes[1, 0].set_ylabel('Accuracy (%)')
axes[1, 0].set_title('MMLU Subject Breakdown')
axes[1, 0].set_xticklabels(subjects, rotation=45, ha='right')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Model ranking
avg_scores = {
    m: np.mean([all_results[m][b] for b in benchmarks])
    for m in model_names
}
sorted_models = sorted(avg_scores.items(), key=lambda x: -x[1])

names_sorted = [m[0] for m in sorted_models]
scores_sorted = [m[1] for m in sorted_models]

bars = axes[1, 1].barh(names_sorted[::-1], scores_sorted[::-1],
                       color=['#FF6B6B', '#FFA07A', '#4ECDC4', '#45B7D1'][::-1],
                       edgecolor='black')
axes[1, 1].set_xlabel('Average Score (%)')
axes[1, 1].set_title('Overall Model Ranking')
axes[1, 1].grid(axis='x', alpha=0.3)

for bar, score in zip(bars, scores_sorted[::-1]):
    axes[1, 1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                   f'{score:.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Statistical Significance Testing

In [ ]:
def bootstrap_confidence_interval(results: List[bool], n_bootstrap: int = 1000,
                                  confidence: float = 0.95) -> Tuple[float, float, float]:
    """Compute confidence interval using bootstrap"""
    n = len(results)
    bootstrapped_accs = []
    
    for _ in range(n_bootstrap):
        sample = np.random.choice(results, size=n, replace=True)
        bootstrapped_accs.append(np.mean(sample))
    
    bootstrapped_accs = np.array(bootstrapped_accs)
    
    alpha = 1 - confidence
    lower = np.percentile(bootstrapped_accs, 100 * alpha / 2)
    upper = np.percentile(bootstrapped_accs, 100 * (1 - alpha / 2))
    mean = np.mean(results)
    
    return mean, lower, upper

# Compute confidence intervals for MMLU
print("📊 MMLU 95% Confidence Intervals:")
print("=" * 50)

for model in models:
    results = [r['correct'] for r in mmlu_results[model.name]['results']]
    mean, lower, upper = bootstrap_confidence_interval(results)
    print(f"{model.name:<15} {mean:.1%} [{lower:.1%}, {upper:.1%}]")

# Error analysis
print("\n📋 Error Analysis (GPT-4 on MMLU):")
gpt4_results = mmlu_results['GPT-4']['results']
wrong = [r for r in gpt4_results if not r['correct']]

if wrong:
    for w in wrong:
        q = next(q for q in mmlu_samples if q.id == w['id'])
        print(f"\n  ❌ {q.id} ({q.subject}, {q.difficulty})")
        print(f"     Predicted: {q.choices[w['predicted']]}")
        print(f"     Correct: {q.choices[w['actual']]}")
else:
    print("  ✅ No errors!")

## 8. Export Results

In [ ]:
# Export results to JSON
export_data = {
    'metadata': {
        'date': '2024-01-01',
        'benchmarks': ['MMLU', 'GSM8K', 'HumanEval'],
        'models': [m.name for m in models],
    },
    'results': {
        'mmlu': {m.name: {
            'accuracy': mmlu_results[m.name]['accuracy'],
            'subject_breakdown': mmlu_results[m.name]['subject_accuracy']
        } for m in models},
        'gsm8k': {m.name: {
            'accuracy': gsm8k_results[m.name]['accuracy'],
            'exact_match': gsm8k_results[m.name]['exact_match']
        } for m in models},
        'humaneval': {m.name: {
            'pass_at_1': humaneval_results[m.name]['pass_at_1'],
            'test_accuracy': humaneval_results[m.name]['test_accuracy']
        } for m in models},
    }
}

with open('benchmark_results.json', 'w') as f:
    json.dump(export_data, f, indent=2)

print("💾 Results exported to benchmark_results.json")

# Summary table
print("\n📊 Final Benchmark Results:")
print("=" * 70)
print(f"{'Model':<15} {'MMLU':<10} {'GSM8K':<10} {'HumanEval':<10} {'Average'}")
print("-" * 70)
for m in model_names:
    scores = [all_results[m][b] for b in benchmarks]
    avg = np.mean(scores)
    print(f"{m:<15} {scores[0]:<10.1f} {scores[1]:<10.1f} {scores[2]:<10.1f} {avg:.1f}")

## 🎯 Exercises

1. **Real Benchmarks**: Use `lm-eval-harness` to evaluate on real MMLU/GSM8K.
2. **Few-shot Prompting**: Implement 0-shot, 1-shot, 5-shot evaluation.
3. **Chain-of-Thought**: Add CoT prompting and evaluate reasoning quality.
4. **Custom Benchmark**: Create a domain-specific benchmark.
5. **Leaderboard**: Build an automated leaderboard comparison.
6. **Error Analysis**: Deep-dive into failure modes per subject.